# Text2SQL over `samples.wanderbricks` — Baseline vs Semantic-Layer Experiment

Runs entirely inside a Databricks workspace against the real `samples.wanderbricks` dataset (built into every Unity Catalog-enabled workspace).

This notebook is structured as a two-section experiment so you can measure the *marginal value* of a semantic layer:

- **Section 1 — Baseline.** The LLM sees the raw schema (from `information_schema.columns`) and nothing else. A tagged battery of questions ranging from trivial aggregates to multi-JOIN, window-function, and business-ambiguous phrasing is run through it. Pass/fail is measured by whether the generated SQL executes; a human still needs to eyeball the result for correctness.
- **Section 2 — With semantic layer.** Same test battery, same prompt structure, but with a YAML semantic layer injected. The YAML starts empty — you populate it based on what you saw fail in Section 1, then re-run to compare.

Both sections use `ChatDatabricks` with `databricks-meta-llama-3-1-8b-instruct` (no external API key needed). To swap in OpenAI, replace the `ChatDatabricks(...)` call and pull the key via `dbutils.secrets.get(...)` — never hardcode.


In [0]:
%pip install -U -q databricks-langchain langchain langchain-core langchain-community pyyaml


In [0]:
dbutils.library.restartPython()


In [0]:
from collections import defaultdict

import pandas as pd
import yaml
from databricks_langchain import ChatDatabricks
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

CATALOG = "samples"
SCHEMA = "wanderbricks"


In [0]:
spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").display()


In [0]:
def build_schema_text(catalog: str, schema: str) -> str:
    """Pull column-level metadata from information_schema and format it for the LLM prompt."""
    rows = spark.sql(f"""
        SELECT table_name, column_name, data_type, ordinal_position
        FROM {catalog}.information_schema.columns
        WHERE table_schema = '{schema}'
        ORDER BY table_name, ordinal_position
    """).collect()

    table_names = []
    table_details = []
    # by_table = defaultdict(list)
    for r in rows:
        table_names.append(r.table_name)
        table_details.append(f"{r.column_name} {r.data_type}")
        # by_table[r.table_name].append(f"{r.column_name} {r.data_type}")

    return pd.DataFrame({"Table":table_names, 
                         "Schema":table_details}).groupby("Table").agg(list).reset_index()
    # return pd.DataFrame([f"Table {catalog}.{schema}.{tbl}: ({', '.join(cols)})" for tbl, cols in by_table.items()], columns=["Schema"])


schema_text = build_schema_text(CATALOG, SCHEMA)
display(schema_text)


In [0]:
# # For each table in samples.wanderbricks: print first 10 rows, summary, and fill rate
# # Collate all per-table information into sheets in a single Excel workbook.

# import os

# import pandas as pd

# # pip install openpyxl if missing
# try:
#     import openpyxl  # noqa: F401
# except ImportError:
#     os.system("pip install -q openpyxl")

# tables = [r.tableName for r in spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").collect()]

# EXCEL_PATH = "/tmp/wanderbricks_profiles.xlsx"

# with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl") as writer:
#     for table_name in tables:
#         full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
#         print("=" * 80)
#         print(f"TABLE: {full_name}")
#         print("=" * 80)

#         df = spark.sql(f"SELECT * FROM {full_name}").toPandas()

#         # --- First 10 rows ---
#         first_10 = df.head(10)
#         print("\n--- First 10 rows ---")
#         display(first_10)

#         # --- Summary (basic stats) ---
#         summary = df.describe()
#         print("\n--- Summary ---")
#         print(summary)

#         # --- Fill rate per column ---
#         fill_rate = (1 - df.isnull().mean()).to_frame("fill_rate")  # non-null fraction
#         print("\n--- Fill rate (non-null %) per column ---")
#         display(fill_rate.T)

#         print("\n")

#         # Sheet name max 31 chars; prefix with index to avoid collisions
#         safe_sheet = table_name[:31]

#         # Write each piece of info into the workbook
#         first_10.to_excel(writer, sheet_name=f"{safe_sheet}_preview", index=False)
#         summary.to_excel(writer, sheet_name=f"{safe_sheet}_summary")
#         fill_rate.to_excel(writer, sheet_name=f"{safe_sheet}_fillrate")

# print(f"\n✅ Excel workbook written to: {EXCEL_PATH}")
# print(f"Sheets: {pd.ExcelFile(EXCEL_PATH).sheet_names}")

# # Download to local machine if running interactively
# try:
#     from google.colab import files  # noqa: F401
#     files.download(EXCEL_PATH)
# except Exception:
#     pass

In [0]:
# import shutil, os

# SRC = "/tmp/wanderbricks_profiles.xlsx"
# DST_DIR = "/Workspace/Users/etisha98@gmail.com/MyWorkspace"
# DST = os.path.join(DST_DIR, "wanderbricks_profiles.xlsx")

# os.makedirs(DST_DIR, exist_ok=True)
# shutil.copy(SRC, DST)
# print(f"✅ Copied {SRC} → {DST}")
# print(f"Size: {os.path.getsize(DST):,} bytes")

In [0]:
llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-1-8b-instruct",
    temperature=0,
)


## Shared test battery

Questions are tagged by difficulty class so the results can be sliced. Categories:

- **simple** — single-table COUNT / filter / order-by; low ambiguity.
- **medium** — GROUP BY, single JOIN, date arithmetic, common aggregates.
- **complex** — multi-JOIN, window functions / LAG, cohort comparisons, running aggregates.
- **ambiguous** — the question can't be answered without a business-defined rule (e.g. what counts as "active", what counts as "profitable"). These are the questions a semantic layer is *supposed* to fix — expect Section 1 to hallucinate or refuse.


In [0]:
test_queries = [
    # ---- simple: single-table aggregates & filters ----
    {"category": "simple", "question": "How many users are registered on the platform?"},
    {"category": "simple", "question": "How many properties are listed?"},
    {"category": "simple", "question": "How many total bookings are there?"},
    {"category": "simple", "question": "How many bookings are in the 'pending' status?"},
    {"category": "simple", "question": "List the 5 most recent bookings by creation date."},

    # ---- medium: grouping, dates, single JOIN ----
    {"category": "medium", "question": "How many bookings does each booking status have?"},
    {"category": "medium", "question": "What is the average nightly price per property type?"},
    {"category": "medium", "question": "How many pending bookings were created in the first six months of 2025?"},
    {"category": "medium", "question": "What are the top 5 destinations by number of properties listed?"},
    {"category": "medium", "question": "Which host has the most property listings?"},
    {"category": "medium", "question": "What is the average number of days between a booking's creation and its check-in date?"},
    {"category": "medium", "question": "Which country has the highest views by ad?"},

    # ---- complex: multi-JOIN, windows, cohort logic ----
    {"category": "complex", "question": "How many bookings has each user made on average?"},
    {"category": "complex", "question": "What is the total gross booking revenue for each month of 2025?"},
    {"category": "complex", "question": "For each property type, what is the average review rating?"},
    {"category": "complex", "question": "Which 5 amenities appear most often in properties that received 5-star reviews?"},
    {"category": "complex", "question": "What is the month-over-month percentage change in total bookings in 2025?"},
    {"category": "complex", "question": "For each user who has booked more than once, compute the average number of days between consecutive bookings."},


    # ---- ambiguous: needs a business definition ----
    {"category": "ambiguous", "question": "How many active users do we have?"},
    {"category": "ambiguous", "question": "What is our booking conversion rate?"},
    {"category": "ambiguous", "question": "Which properties are our top performers?"},
    {"category": "ambiguous", "question": "What is the most used amenities by 5 star customers from each country?"},
    {"category": "ambiguous", "question": "How many defaulters have given 5 star ratings and have visited Agra?"},
    {"category": "ambiguous", "question": "How many users have visited more than 10 countries?"},
    {"category": "ambiguous", "question": "What is the average number of bookings per user?"},
    {"category": "ambiguous", "question": "Which is the most preferred host in India and China? What are their ratings?"},
    {"category": "ambiguous", "question": "What is the average rating of hosts and properties in each country?"},
    {"category": "ambiguous", "question": "Show the names and email addresses of all hosts who are also registered as users."},
    {"category": "ambiguous", "question": "Which destinations have the most properties listed?"},
    {"category": "ambiguous", "question": "Which properties in Spain have an average rating higher than 4.5?"},
    {"category": "ambiguous", "question": "How many active users signed up in the last 6 months?"}, 
    {"category": "ambiguous", "question": "Find the total discount amount applied to bookings in 2025"},
    {"category": "ambiguous", "question": "What is the average review score for properties located in the city of Paris?"},
    {"category": "ambiguous", "question": "Which device type generated the highest number of clickstream events?"},
    {"category": "ambiguous", "question": "Show the country code for all clickstream interactions."},
    {"category": "ambiguous", "question": "Which amenities are most frequently found in Ski Resort properties?"},
    {"category": "ambiguous", "question": "How many users have booked properties outside their own country?"},
    {"category": "ambiguous", "question": "What is the average review rating for properties that have a primary photo?"},
    {"category": "ambiguous", "question": "What is the average guest review rating for each property type?"},
    {"category": "ambiguous", "question": "What is the average lead time between booking creation and check-in for confirmed stays?"}
]

len(test_queries)


In [0]:
def run_battery(ask_fn, queries):
    """Execute every question, capture generated SQL, result preview, and any error."""
    rows = []
    for q in queries:
        out = ask_fn(q["question"])
        preview = (
            out["result"].head(3).to_dict("records")
            if out["result"] is not None else None
        )
        rows.append({
            "category": q["category"],
            "question": q["question"],
            "executed_ok": out["error"] is None,
            "row_count": out["row_count"],
            "sample_result": preview,
            "error": out["error"],
            "generated_sql": out["query"],
        })
    return pd.DataFrame(rows)


def summarize(results_df: pd.DataFrame) -> pd.DataFrame:
    return (
        results_df.groupby("category")
        .agg(
            total=("executed_ok", "count"),
            passed=("executed_ok", "sum"),
            pass_rate=("executed_ok", "mean"),
        )
        .reset_index()
    )


## Section 1 — Baseline: Text2SQL with schema-only context

The LLM sees only the auto-derived schema. No metrics, no taxonomies, no business rules. This is the null hypothesis: how far does a modern instruction-tuned model get on the wanderbricks dataset armed only with column names and types?

**What to watch for when reviewing results:**

1. **Hard failures (SQL errors)** — column-name hallucination, wrong function names, invalid syntax. `executed_ok = False` catches these.
2. **Silent failures** — SQL runs but result is wrong. Skim `sample_result` and check whether the numbers make sense. Rows with `row_count == 0` on a question that should have data are prime suspects.
3. **Ambiguous questions** — the LLM will happily invent a definition (e.g., "active user" = signed up in last 30 days). Note which definitions it picks; those become candidates for taxonomy entries in Section 2.
4. **Complex questions that pass by accident** — the SQL might be correct but inefficient or fragile. Worth a second look.


In [0]:
system_template_bare = """You are an expert Databricks SQL developer working against the Unity Catalog dataset samples.wanderbricks.

Generate a valid Databricks SQL query that answers the user question using ONLY the schema below.

Schema:
{schema}

Guidelines:
- Return ONLY the executable SQL query. No markdown fences, no explanations.
- Use standard Databricks SQL syntax.
- Always reference tables with their full three-part name (samples.wanderbricks.<table>).
- Use JOINs when the answer spans multiple tables."""

prompt_bare = ChatPromptTemplate.from_messages([
    ("system", system_template_bare),
    ("human", "Question: {question}"),
])


def ask_bare(question: str) -> dict:
    chain = prompt_bare | llm | StrOutputParser()
    raw_sql = chain.invoke({"schema": schema_text, "question": question})
    clean_sql = raw_sql.replace("```sql", "").replace("```", "").strip()
    try:
        df = spark.sql(clean_sql).toPandas()
        return {"query": clean_sql, "result": df, "error": None, "row_count": len(df)}
    except Exception as exc:
        return {"query": clean_sql, "result": None, "error": str(exc)[:300], "row_count": None}


In [0]:
# Prevent truncation of rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None) 

In [0]:
results_bare = run_battery(ask_bare, test_queries)
display(results_bare)

In [0]:
summarize(results_bare)


In [0]:
## Ambiguous queries
ambiguous_bare = results_bare[(results_bare["category"] == "ambiguous")]# & (results_bare["error"].notna())]
ambiguous_bare

In [0]:
spark.sql("SELECT a.name FROM samples.wanderbricks.property_amenities pa JOIN samples.wanderbricks.amenities a ON pa.amenity_id = a.amenity_id JOIN samples.wanderbricks.reviews r ON pa.property_id = r.property_id WHERE r.rating = 5 GROUP BY a.name ORDER BY COUNT(pa.property_id) DESC LIMIT 5").toPandas()

In [0]:
# Failed queries only — inspect the generated SQL and error to see what broke
failed_bare = results_bare[~results_bare["executed_ok"]][["category", "question", "generated_sql", "error"]]
failed_bare


In [0]:
# Drill into one query at a time (change the index). Prints the SQL and shows the result frame.
IDX = 25  # e.g. "How many pending bookings were created in the first six months of 2025?"

row = results_bare.iloc[IDX]
print(f"Q: {row['question']}")
print(f"Category: {row['category']}  |  executed_ok: {row['executed_ok']}  |  row_count: {row['row_count']}")
print("\n--- Generated SQL ---")
print(row["generated_sql"])
if row["error"]:
    print("\n--- Error ---")
    print(row["error"])
else:
    print("\n--- Sample result ---")
    print(row["sample_result"])


## Section 2 — Text2SQL with semantic layer

**Populate the YAML below after reviewing Section 1's failures.** The scaffold below is intentionally minimal — the retriever, prompt, and runner are all wired up, so all you need to iterate on is `semantic_layer_yaml`.

Suggested workflow:

1. Look at every `ambiguous` failure in Section 1 → add a **taxonomy** or **metric** that pins down the business definition.
2. Look at every `complex` query where the SQL was subtly wrong (e.g. wrong join key, wrong date column) → add an **entity attribute** clarifying which column carries which meaning, or a **metric** with a canonical SQL snippet.
3. Re-run the battery; compare `pass_rate` and eyeball the generated SQL for the ambiguous questions to see whether they now use your definitions.


In [0]:
spark.sql("""SELECT status, COUNT(*) AS n
FROM samples.wanderbricks.payments
GROUP BY status
ORDER BY n DESC""").display()

In [0]:
# Empty starting point — populate this after reviewing Section 1 results.
# Same shape as the local notebook's semantic layer: entities / metrics / taxonomies.
semantic_layer_yaml = """
version: "1.0"
domain: "wanderbricks_marketplace"

entities:
  - name: "User"
    table: "users"
    description: "A registered platform user (traveler)"

  - name: "Property"
    table: "properties"
    description: "A listed rental property"
    attributes:
      - name: "destination_id"
        column: "destination_id"
        type: "STRING"
        description: "FK to destinations.destination_id. NOTE: destinations also has a column named destination_id — always qualify with table alias when both tables are in the same query"

metrics:
  active_users:
    definition: "Users with 2 or more bookings in the trailing 12 months, anchored to the latest data date (not current_date())"
    sql: "COUNT(DISTINCT user_id) FROM bookings WHERE created_at >= (SELECT MAX(created_at) FROM bookings) - INTERVAL 12 MONTHS GROUP BY user_id HAVING COUNT(*) >= 2"

  inactive_users:
    definition: "Users with zero bookings in the trailing 12 months"
    sql: "SELECT user_id FROM users WHERE user_id NOT IN (SELECT DISTINCT user_id FROM bookings WHERE created_at >= (SELECT MAX(created_at) FROM bookings) - INTERVAL 12 MONTHS)"

  active_properties:
    definition: "Properties with 2 or more bookings in the trailing 12 months — mirrors active_users"
    sql: "COUNT(DISTINCT property_id) FROM bookings WHERE created_at >= (SELECT MAX(created_at) FROM bookings) - INTERVAL 12 MONTHS GROUP BY property_id HAVING COUNT(*) >= 2"

  total_registered_users:
    definition: "All users regardless of activity"
    sql: "COUNT(*) FROM users"

  gross_booking_volume:
    definition: "Total count of bookings, all statuses"
    sql: "COUNT(*) FROM bookings"

  confirmed_booking_revenue:
    definition: "Total value of confirmed bookings"
    sql: "SUM(total_amount) FROM bookings WHERE status = 'confirmed'"

  booking_status_distribution:
    definition: "Percentage share of bookings by status"
    sql: "COUNT(*) / SUM(COUNT(*)) OVER () FROM bookings GROUP BY status"

  booking_lead_time:
    definition: "Average days between booking creation and check-in"
    sql: "AVG(DATEDIFF(check_in_date, created_at)) FROM bookings"

  mom_booking_growth:
    definition: "Month-over-month percentage change in booking count"
    sql: "(current_month_count - previous_month_count) / previous_month_count -- requires monthly aggregation subquery with LAG() OVER (ORDER BY month)"

  avg_bookings_per_user:
    definition: "Average number of bookings per registered user"
    sql: "COUNT(*) FROM bookings / COUNT(DISTINCT user_id) FROM users"

  repeat_guest_rate:
    definition: "Percentage of users with more than 1 completed booking"
    sql: "COUNT(DISTINCT CASE WHEN booking_count > 1 THEN user_id END) / COUNT(DISTINCT user_id) FROM (SELECT user_id, COUNT(*) AS booking_count FROM bookings WHERE status = 'completed' GROUP BY user_id)"

  avg_review_rating_by_property_type:
    definition: "Average review rating grouped by property type"
    sql: "AVG(reviews.rating) FROM reviews JOIN properties ON reviews.property_id = properties.property_id GROUP BY properties.property_type"

  average_nightly_rate_by_type:
    definition: "Average nightly price grouped by property type — CONFIRM properties has a price/rate column before using"
    sql: "AVG(price) FROM properties GROUP BY property_type"

  booking_conversion_rate:
    definition: "Share of users with a clickstream 'view' or 'search' event who went on to complete a booking"
    sql: "COUNT(DISTINCT bookings.user_id) / COUNT(DISTINCT clickstream.user_id) WHERE clickstream.event IN ('view','search')"

  super_traveler_ratio:
    definition: "Users with completed bookings across properties in more than 10 distinct destination countries." 
    sql: |
      WITH super_travelers AS (
        SELECT b.user_id
        FROM samples.wanderbricks.bookings b
        JOIN samples.wanderbricks.properties p    ON b.property_id = p.property_id
        JOIN samples.wanderbricks.destinations d  ON p.destination_id = d.destination_id
        WHERE b.status = 'completed'
        GROUP BY b.user_id
        HAVING COUNT(DISTINCT d.country) > 10
      )
      SELECT
        COUNT(DISTINCT st.user_id) / COUNT(DISTINCT u.user_id) AS super_traveler_ratio,
        COUNT(DISTINCT st.user_id) AS super_traveler_count,
        COUNT(DISTINCT u.user_id) AS total_users
      FROM samples.wanderbricks.users u
      LEFT JOIN super_travelers st ON u.user_id = st.user_id

  repeat_host_rate:
    definition: "what share of users are host-loyal."
    sql: "COUNT(DISTINCT user_id) FROM (<your existing query>) / COUNT(DISTINCT user_id) FROM users"

  listing_performance_score:
    definition: "Weighted average of a listing's review ratings (1,2->0.2; 3->0.3; 4->0.4; 5->0.5), scaled by log(1 + review_count) so listings with more reviews rank higher, with diminishing returns as review count grows."
    sql: |
      SELECT
        property_id,
        SUM(CASE rating
              WHEN 1 THEN 0.2 WHEN 2 THEN 0.2
              WHEN 3 THEN 0.3 WHEN 4 THEN 0.4
              WHEN 5 THEN 0.5
            END) / COUNT(*)                          AS weighted_rating_score,
        COUNT(*)                                       AS review_count,
        (SUM(CASE rating
              WHEN 1 THEN 0.2 WHEN 2 THEN 0.2
              WHEN 3 THEN 0.3 WHEN 4 THEN 0.4
              WHEN 5 THEN 0.5
            END) / COUNT(*)) * LN(1 + COUNT(*))       AS weighted_rating_score_volume_adjusted
      FROM samples.wanderbricks.reviews
      GROUP BY property_id

  defaulter_rate:
    definition: "User has ≥1 payment row with status = 'failed', regardless of what happened after"
    sql: |
      WITH booking_payment_status AS (
        SELECT
          b.user_id,
          b.booking_id,
          MAX(CASE WHEN p.status = 'completed' THEN 1 ELSE 0 END) AS ever_paid,
          MAX(CASE WHEN p.status = 'failed' THEN 1 ELSE 0 END)    AS has_failed_attempt
        FROM samples.wanderbricks.bookings b
        JOIN samples.wanderbricks.payments p ON b.booking_id = p.booking_id
        GROUP BY b.user_id, b.booking_id
      ),
      defaulted_users AS (
        SELECT DISTINCT user_id
        FROM booking_payment_status
        WHERE has_failed_attempt = 1 AND ever_paid = 0
      )
      SELECT
        COUNT(DISTINCT d.user_id) / COUNT(DISTINCT u.user_id) AS payment_default_rate,
        COUNT(DISTINCT d.user_id) AS defaulter_count,
        COUNT(DISTINCT u.user_id) AS total_users
      FROM samples.wanderbricks.users u
      LEFT JOIN defaulted_users d ON u.user_id = d.user_id

taxonomies:
  booking_status:
    column: "status"
    mappings:
      pending: "Booking created, awaiting host or payment confirmation"
      confirmed: "Booking is confirmed and counts toward revenue"
      cancelled: "Booking was cancelled — excluded from revenue metrics"
      completed: "Stay has occurred"

  property_type:
    column: "property_type"
    mappings:
      apartment: "Apartment/flat listing"
      villa: "Standalone villa"
      cabin: "Cabin or cottage"
      hotel: "Hotel room listing"

  clickstream_event:
    column: "event"
    mappings:
      view: "Property listing viewed"
      search: "Search performed"
      ad_view: "Advertisement impression viewed"
      click: "Generic click event"
"""

semantic_data = yaml.safe_load(semantic_layer_yaml)

In [0]:
def retrieve_semantic_context(query: str) -> str:
    """Return metric + taxonomy definitions relevant to the user question via keyword match."""
    chunks = []
    q = query.lower()

    for name, det in (semantic_data.get("metrics") or {}).items():
        readable = name.replace("_", " ")
        keyword_hit = any(w in q for w in name.split("_") if len(w) > 3)
        if readable in q or keyword_hit:
            chunks.append(
                f"Metric '{name}':\n"
                f"- Definition: {det['definition']}\n"
                f"- Canonical SQL: {det['sql']}"
            )

    for name, det in (semantic_data.get("taxonomies") or {}).items():
        readable = name.replace("_", " ")
        mappings = det.get("mappings", {})
        if readable in q or any(k in q for k in mappings.keys()):
            m_str = "; ".join(f"'{k}' = {v}" for k, v in mappings.items())
            chunks.append(
                f"Taxonomy for column '{det['column']}' in table '{det.get('table', 'N/A')}':\n"
                f"- Values: {m_str}"
            )

    return "\n\n---\n\n".join(chunks)


In [0]:
system_template_semantic = """You are an expert Databricks SQL developer working against the Unity Catalog dataset samples.wanderbricks.

Generate a valid Databricks SQL query that answers the user question using the schema and business rules below.

Schema:
{schema}

Business rules and semantic layer:
{semantic_context}

Guidelines:
- Return ONLY the executable SQL query. No markdown fences, no explanations.
- Use standard Databricks SQL syntax.
- Always reference tables with their full three-part name (samples.wanderbricks.<table>).
- Prefer the metric SQL formulas from the semantic layer when they apply.
- Use JOINs when the answer spans multiple tables."""

prompt_semantic = ChatPromptTemplate.from_messages([
    ("system", system_template_semantic),
    ("human", "Question: {question}"),
])


def ask_with_semantic(question: str) -> dict:
    semantic_context = (
        retrieve_semantic_context(question)
        or "No matching semantic rules for this question."
    )
    chain = prompt_semantic | llm | StrOutputParser()
    raw_sql = chain.invoke({
        "schema": schema_text,
        "semantic_context": semantic_context,
        "question": question,
    })
    clean_sql = raw_sql.replace("```sql", "").replace("```", "").strip()
    try:
        df = spark.sql(clean_sql).toPandas()
        return {"query": clean_sql, "result": df, "error": None, "row_count": len(df)}
    except Exception as exc:
        return {"query": clean_sql, "result": None, "error": str(exc)[:300], "row_count": None}


In [0]:
results_semantic = run_battery(ask_with_semantic, test_queries)
results_semantic


In [0]:
# Side-by-side: pass rate before vs after semantic layer, per category
comparison = (
    summarize(results_bare).rename(columns={"passed": "passed_bare", "pass_rate": "pass_rate_bare"})
    .merge(
        summarize(results_semantic).rename(columns={"passed": "passed_sem", "pass_rate": "pass_rate_sem"}),
        on=["category", "total"],
        how="outer",
    )
)
comparison


In [0]:
# Rows where semantic layer changed the outcome (passed one way, failed the other, or produced different SQL)
diff = results_bare[["question", "executed_ok", "generated_sql"]].rename(
    columns={"executed_ok": "ok_bare", "generated_sql": "sql_bare"}
).merge(
    results_semantic[["question", "executed_ok", "generated_sql"]].rename(
        columns={"executed_ok": "ok_sem", "generated_sql": "sql_sem"}
    ),
    on="question",
)
diff["sql_changed"] = diff["sql_bare"] != diff["sql_sem"]
diff[(diff["ok_bare"] != diff["ok_sem"])]
